# Native second-quantized Hamiltonian

Retain the source four modes and three ordered fermionic monomials. Use FlagQuantum's own fermionic algebra and Jordan–Wigner mapping, then synthesize a three-layer native ansatz and evaluate its observable. No OpenFermion or Classiq SDK is required. The last source term has positive occupation energy after anticommutation: -0.3 c†2 c†3 c2 c3 = +0.3 n2 n3.


In [ ]:
from examples.synthesis.applications.chemistry.second_quantized_hamiltonian.overview.optimization import (
    hamiltonian,
)

observable = hamiltonian()
print([(term.coefficient, term.factors) for term in observable.terms])


## One composable native quantum function

Keep the exact source layer order: X on each qubit followed by its RY rotation, then a nearest-neighbor CX chain. Three layers consume twelve angles. The source records particle metadata (1, 1), but this ansatz does not preserve particle number. The following optimization searches the full 16-dimensional space and reports observed particle populations; it is not a fixed-sector ground-state solver.


In [ ]:
import math

from flagquantum import synth
from flagquantum.synth import CX, Output, QArray, QBit, R, X, allocate, qfunc


@qfunc
def variational_program(params: list[float], state: Output[QArray[QBit]]) -> None:
    allocate(4, state)
    for layer in range(3):
        for index in range(4):
            X(state[index])
            R(params[4 * layer + index], math.pi / 2, state[index])
        for index in range(3):
            CX(state[index], state[index + 1])


program = synth.synthesize(
    variational_program,
    classical_inputs={"params": [0.0] * 12},
    max_qubits=4,
    max_workspace=0,
)
print(program.report["resources"])


## Actual energy optimization

Start from twelve zero angles. Each objective evaluation synthesizes the native ansatz and uses fq.expectation on the mapped observable. An explicit coordinate search is bounded by 200 evaluations; this is a documented replacement for an unspecified private optimizer with 200 iterations, without a solver-equivalence claim. No known spectrum or target energy is passed to the optimizer. Retain all candidate energies, including rejected candidates. Budget exhaustion does not certify convergence.


In [ ]:
from examples.synthesis.applications.chemistry.second_quantized_hamiltonian.overview.analysis import (
    run_workflow,
)

workflow = run_workflow()
print("Initial energy:", workflow.optimization.initial_energy)
print("Best visited energy:", workflow.energy)
print("Evaluations:", len(workflow.optimization.trace))
print("Termination:", workflow.optimization.termination)
print("Particle-number populations:", dict(workflow.sectors))
assert len(workflow.optimization.trace) == 200
assert workflow.energy < workflow.optimization.initial_energy


## Independent validation

Apply the ordered fermion operators to occupation states from right to left, with parity signs, to build an independent 16×16 matrix. Execute an independent Circuit ansatz and compare full complex amplitudes, without phase alignment. Use spectral enumeration only here as validation, never as optimizer input. The printed energy is variational evidence from a local exact simulation; no hardware, speedup, fixed-sector or global convergence claim is made.


In [ ]:
import torch

import flagquantum as fq
from examples.synthesis.applications.chemistry.second_quantized_hamiltonian.overview.matrix import (
    occupation_matrix,
    reference_circuit,
)

reference = (
    fq.run(reference_circuit(workflow.optimization.parameters)).statevector().flatten()
)
actual = fq.run(workflow.program).statevector().flatten()
torch.testing.assert_close(actual, reference, atol=1e-12, rtol=0)
matrix = occupation_matrix()
independent_energy = float((reference.conj() @ (matrix @ reference)).real)
assert abs(workflow.energy - independent_energy) < 1e-12
print("Independent energy:", independent_energy)
print("Full-space spectrum (validation only):", torch.linalg.eigvalsh(matrix).tolist())
print("Maximum full-state error:", float((actual - reference).abs().max()))
